# SQL Analysis of COVID-19 Health Data

In this notebook, I use SQL to analyze cleaned COVID-19 health data.  
The goal is to answer analytical questions about cases, deaths, vaccination progress, and country-level comparisons.

In [127]:
import pandas as pd
import sqlite3

In [128]:
df = pd.read_csv("../data/clean_data_4countries.csv")

In [129]:
df.head

<bound method NDFrame.head of            location        date  population  new_cases  new_deaths  \
0             China  2020-01-05  1425887360        1.0         0.0   
1             China  2020-01-06  1425887360        0.0         0.0   
2             China  2020-01-07  1425887360        0.0         0.0   
3             China  2020-01-08  1425887360        0.0         0.0   
4             China  2020-01-09  1425887360        0.0         0.0   
...             ...         ...         ...        ...         ...   
6691  United States  2024-07-31   338289856        0.0         0.0   
6692  United States  2024-08-01   338289856        0.0         0.0   
6693  United States  2024-08-02   338289856        0.0         0.0   
6694  United States  2024-08-03   338289856        0.0         0.0   
6695  United States  2024-08-04   338289856        0.0       619.0   

      total_cases  total_deaths  people_vaccinated  total_vaccinations  \
0             1.0           0.0                NaN     

In [130]:
df.shape

(6696, 15)

In [131]:
df.columns

Index(['location', 'date', 'population', 'new_cases', 'new_deaths',
       'total_cases', 'total_deaths', 'people_vaccinated',
       'total_vaccinations', 'new_cases_per_million', 'new_deaths_per_million',
       'total_deaths_per_million', 'new_cases_7day_avg', 'new_deaths_7day_avg',
       'vaccinated_percentage'],
      dtype='object')

In [132]:
# create a database for sql
conn = sqlite3.connect("../data/covid_health.db")

In [133]:
df.to_sql("covid_table", conn, if_exists = "replace", index = False)

6696

In [134]:
query = """
SELECT *
FROM covid_table
LIMIT 5;
"""

pd.read_sql_query(query,conn) # Use pandas to execute the sql query from database, return the results in dateframe

,location,date,population,new_cases,new_deaths,total_cases,total_deaths,people_vaccinated,total_vaccinations,new_cases_per_million,new_deaths_per_million,total_deaths_per_million,new_cases_7day_avg,new_deaths_7day_avg,vaccinated_percentage
0,China,2020-01-05,1425887360,1.0,0.0,1.0,0.0,None,None,0.000701,0.0,0.0,0.000701,0.0,None
1,China,2020-01-06,1425887360,0.0,0.0,1.0,0.0,None,None,0.000000,0.0,0.0,0.000351,0.0,None
2,China,2020-01-07,1425887360,0.0,0.0,1.0,0.0,None,None,0.000000,0.0,0.0,0.000234,0.0,None
3,China,2020-01-08,1425887360,0.0,0.0,1.0,0.0,None,None,0.000000,0.0,0.0,0.000175,0.0,None
4,China,2020-01-09,1425887360,0.0,0.0,1.0,0.0,None,None,0.000000,0.0,0.0,0.000140,0.0,None


In [135]:
# check the sql table structure
query = """
PRAGMA table_info(covid_table)
"""
pd.read_sql_query(query,conn)

,cid,name,type,notnull,dflt_value,pk
0,0,location,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,population,INTEGER,0,None,0
3,3,new_cases,REAL,0,None,0
4,4,new_deaths,REAL,0,None,0
5,5,total_cases,REAL,0,None,0
6,6,total_deaths,REAL,0,None,0
7,7,people_vaccinated,REAL,0,None,0
8,8,total_vaccinations,REAL,0,None,0
9,9,new_cases_per_million,REAL,0,None,0


# query 1: how many records each country has?

In [137]:
query = """
SELECT 
    location, 
    COUNT(*) AS number_records
FROM covid_table
GROUP BY location
ORDER BY number_records DESC;
"""
pd.read_sql_query(query,conn)

,location,number_records
0,United States,1674
1,Japan,1674
2,Germany,1674
3,China,1674


This query checks how many daily records are available for each country.  
It helps identify whether some countries have much shorter or incomplete time series.

# query 2: The highest number of daily new cases of each country

In [140]:
query = """
SELECT 
    location,
    MAX(new_cases) AS max_new_cases_daily
FROM covid_table
GROUP BY location
ORDER BY max_new_cases_daily DESC;
"""
pd.read_sql_query(query,conn)


,location,max_new_cases_daily
0,China,40475477.0
1,United States,5650933.0
2,Germany,1588891.0
3,Japan,1496968.0


# query 3: Compare the countries by the highest number of daily new cases per million people

In [142]:
query = """
SELECT
    location,
    MAX(new_cases_per_million) AS max_new_cases_per_million
FROM covid_table
GROUP BY location
ORDER BY max_new_cases_per_million DESC;
"""
pd.read_sql_query(query,conn)

,location,max_new_cases_per_million
0,China,28386.167193
1,Germany,19058.342921
2,United States,16704.411616
3,Japan,12077.027167


# query 4: The largest cumulative deaths of each country

In [144]:
query = """
SELECT 
    location,
    max(total_deaths) AS total_deaths_latest
FROM covid_table
GROUP BY location
ORDER BY total_deaths_latest DESC;
"""
pd.read_sql_query(query,conn)

,location,total_deaths_latest
0,United States,1193165.0
1,Germany,174979.0
2,China,122304.0
3,Japan,74694.0


# query 5: Compare the countries by the death rate proxy: total_deaths/total_cases

In [146]:
query = """
SELECT
    location,
    MAX(total_deaths) AS total_deaths_latest,
    MAX(total_cases) AS total_cases_latest,
    ROUND(
        MAX(total_deaths)*100.0/NULLIF(MAX(total_cases),0),
        2
        ) AS fatality_rate_percent
FROM covid_table
GROUP BY location
HAVING total_cases_latest>100000
ORDER BY  fatality_rate_percent DESC
"""

pd.read_sql_query(query,conn)

,location,total_deaths_latest,total_cases_latest,fatality_rate_percent
0,United States,1193165.0,103436829.0,1.15
1,Germany,174979.0,38437756.0,0.46
2,Japan,74694.0,33803572.0,0.22
3,China,122304.0,99373219.0,0.12


This is only a proxy and should not be interpreted as the true infection fatality rate, because confirmed cases depend on testing capacity, reporting practices, and policy differences.

# query 6: Compare countries by the highest reported percentage of vacinnated people

In [149]:
query = """
SELECT 
    location,
    MAX(vaccinated_percentage) AS max_vaccinated_percentage
FROM covid_table
GROUP BY location
ORDER BY max_vaccinated_percentage DESC
"""
pd.read_sql_query(query,conn)

,location,max_vaccinated_percentage
0,China,91.893093
1,Japan,84.472530
2,United States,79.880368
3,Germany,77.817469


# query 7: Compare countries using cumulative deaths,cumulative cases, peak daily cases per million and average daily cases per million

In [151]:
query = """
SELECT
    location,
    COUNT(*) AS num_days, 
    MAX(total_deaths) AS lastest_total_deaths,
    MAX(total_cases) AS lastest_total_cases,
    MAX(new_cases_per_million) AS max_new_cases_per_million,
    AVG(new_cases_per_million) AS avg_new_cases_per_million
FROM covid_table
WHERE location IN ('United States', 'Germany', 'China', 'Japan')
GROUP BY location
ORDER BY lastest_total_cases DESC;
"""   
country_summary = pd.read_sql_query(query,conn)
country_summary

,location,num_days,lastest_total_deaths,lastest_total_cases,max_new_cases_per_million,avg_new_cases_per_million
0,United States,1674,1193165.0,103436829.0,16704.411616,182.654665
1,China,1674,122304.0,99373219.0,28386.167193,41.632135
2,Germany,1674,174979.0,38437756.0,19058.342921,275.418814
3,Japan,1674,74694.0,33803572.0,12077.027167,162.912597


In [152]:
country_summary.to_csv("../data/sql_country_summary.csv", index = False)

# Query 8: Compare how the pandemic envolved across different years and countries 

In [154]:
query = """
SELECT
    location,
    strftime("%Y",date) AS year,
    SUM(new_cases) AS yearly_new_cases,
    SUM(new_deaths) AS yearly_new_deaths
FROM covid_table
GROUP BY location, year
ORDER BY location, year;
"""
pd.read_sql_query(query,conn)

,location,year,yearly_new_cases,yearly_new_deaths
0,China,2020,96324.0,4777.0
1,China,2021,34534.0,922.0
2,China,2022,62314792.0,33296.0
3,China,2023,36877077.0,82898.0
4,China,2024,50492.0,433.0
5,Germany,2020,1660178.0,47009.0
6,Germany,2021,5353865.0,70683.0
7,Germany,2022,30227893.0,48046.0
8,Germany,2023,1195820.0,9241.0
9,Germany,2024,0.0,0.0


In [155]:
conn.close()

## Key Findings

1. Python/pandas and SQL served different roles in the workflow. Pandas was more useful for cleaning the data and visualizing time-series trends, while SQL was more useful for producing structured summary tables and country-level comparisons.

2. Raw counts and population-adjusted metrics answer different analytical questions. Raw case counts reflect the overall national burden, while per-million metrics provide a fairer basis for comparing outbreak intensity across countries.

3. The analysis focuses on four large countries with relatively established reporting systems, which makes the comparison easier to interpret. However, the results should not be generalized to all countries without additional checks on missingness, sample size, reporting quality, and policy differences.

4. Yearly aggregation reduces the noise of daily reporting and provides a higher-level view of pandemic phases. It helps identify which years carried the largest reported burden in each selected country.

5. A reproducible workflow connects raw data, cleaned data, SQL queries, summary tables, visualizations, and written interpretation. This structure makes the analysis easier to review, update, and extend.